# Proyecto Final de Minería de Datos — Fase de Despliegue con Interfaz Gráfica Streamlit (CRISP-DM)
## Dataset: Predicción de Abandono de Clientes de Tarjetas de Crédito (*Credit Card Churn*)

Este notebook documenta y ejecuta la **Fase 6 (Despliegue)** de la metodología **CRISP-DM**:
1. **Inspección y Verificación del `Pipeline` Empaquetado (`modelo_churn_pipeline.joblib`)**
2. **Simulación de Inferencia sobre Nuevos Perfiles de Clientes (Datos Futuros)**
3. **Arquitectura de la Aplicación Interactiva en Streamlit (`app_streamlit.py`)**
4. **Plan de Monitoreo de Deriva de Datos (*Data Drift* / *Concept Drift*) y Cronograma de Mantenimiento**


In [1]:
import joblib
import pandas as pd
import numpy as np

bundle = joblib.load("modelo_churn_pipeline.joblib")
pipeline = bundle["pipeline"]

print("=== PIPELINE CARGADO EXITOSAMENTE PARA PRODUCCIÓN ===")
print(f"Pasos del Pipeline: {[step[0] for step in pipeline.steps]}")
print(f"Método de optimización ganador: {bundle['optimization_method']}")
print(f"Hiperparámetros óptimos del clasificador XGBoost: {bundle['best_params']}")
print(f"Métricas validadas en Test (30%): {bundle['test_metrics']}")


=== PIPELINE CARGADO EXITOSAMENTE PARA PRODUCCIÓN ===
Pasos del Pipeline: ['preprocessor', 'smote', 'classifier']
Método de optimización ganador: Optimización Bayesiana (Optuna)
Hiperparámetros óptimos del clasificador XGBoost: {'n_estimators': 250, 'max_depth': 5, 'learning_rate': 0.08046552389871331, 'subsample': 0.8640130838029838, 'colsample_bytree': 0.7554563366576581}
Métricas validadas en Test (30%): {'Accuracy': 0.9651, 'Precision': np.float64(0.8946), 'Recall': np.float64(0.8873), 'F1-Score': np.float64(0.8909), 'ROC-AUC': np.float64(0.9911)}


In [2]:
def add_engineered_features(data):
    d = data.copy()
    d['Avg_Ticket_Size'] = d['Total_Trans_Amt'] / np.maximum(d['Total_Trans_Ct'], 1)
    d['Inactivity_Contact_Ratio'] = d['Months_Inactive_12_mon'] * d['Contacts_Count_12_mon']
    d['Products_Per_Year'] = d['Total_Relationship_Count'] / (d['Months_on_book'] / 12.0)
    return d

nuevos_clientes = pd.DataFrame([
    {
        "Perfil": "Cliente A (Activo y Vinculado)",
        "Customer_Age": 45, "Gender": "M", "Dependent_count": 2,
        "Education_Level": "Graduate", "Marital_Status": "Married",
        "Income_Category": "$80K - $120K", "Card_Category": "Blue",
        "Months_on_book": 36, "Total_Relationship_Count": 5,
        "Months_Inactive_12_mon": 1, "Contacts_Count_12_mon": 1,
        "Credit_Limit": 12000.0, "Total_Revolving_Bal": 1450,
        "Total_Amt_Chng_Q4_Q1": 0.85, "Total_Trans_Amt": 4800,
        "Total_Trans_Ct": 78, "Total_Ct_Chng_Q4_Q1": 0.82,
        "Avg_Utilization_Ratio": 0.121
    },
    {
        "Perfil": "Cliente B (Baja Actividad y Múltiples Reclamos)",
        "Customer_Age": 48, "Gender": "F", "Dependent_count": 3,
        "Education_Level": "High School", "Marital_Status": "Single",
        "Income_Category": "Less than $40K", "Card_Category": "Blue",
        "Months_on_book": 39, "Total_Relationship_Count": 2,
        "Months_Inactive_12_mon": 4, "Contacts_Count_12_mon": 4,
        "Credit_Limit": 3200.0, "Total_Revolving_Bal": 0,
        "Total_Amt_Chng_Q4_Q1": 0.42, "Total_Trans_Amt": 1650,
        "Total_Trans_Ct": 32, "Total_Ct_Chng_Q4_Q1": 0.38,
        "Avg_Utilization_Ratio": 0.000
    }
])

X_nuevos = add_engineered_features(nuevos_clientes.drop(columns=["Perfil"]))
preds = pipeline.predict(X_nuevos)
probs = pipeline.predict_proba(X_nuevos)[:, 1]

res_inferencia = nuevos_clientes[["Perfil", "Total_Trans_Ct", "Total_Revolving_Bal", "Months_Inactive_12_mon", "Contacts_Count_12_mon"]].copy()
res_inferencia["Predicción"] = np.where(preds == 1, "Attrited Customer (Abandona)", "Existing Customer (Retenido)")
res_inferencia["Probabilidad_Churn (%)"] = np.round(probs * 100, 2)
print("=== PRUEBA DE INFERENCIA EN DATOS FUTUROS ===")
print(res_inferencia.to_string(index=False))


=== PRUEBA DE INFERENCIA EN DATOS FUTUROS ===
                                         Perfil  Total_Trans_Ct  Total_Revolving_Bal  Months_Inactive_12_mon  Contacts_Count_12_mon                   Predicción  Probabilidad_Churn (%)
                 Cliente A (Activo y Vinculado)              78                 1450                       1                      1 Existing Customer (Retenido)                0.010000
Cliente B (Baja Actividad y Múltiples Reclamos)              32                    0                       4                      4 Attrited Customer (Abandona)               99.959999


---
## 3. Ejecución de la Interfaz Gráfica Web en Streamlit (`app_streamlit.py`)

La aplicación de despliegue se encuentra implementada en el archivo `app_streamlit.py` e incluye:
- **Pestaña 1 — Predicción Individual en Tiempo Real**: Formulario interactivo dividido en *Datos Demográficos*, *Relación con el Banco* y *Comportamiento Transaccional*, cálculo automático de variables de ingeniería, probabilidad estimada de fuga y recomendación comercial personalizada.
- **Pestaña 2 — Predicción Masiva por Lote (CSV)**: Carga de archivos `.csv` de clientes para calificación masiva y descarga inmediata de resultados.

Para iniciar la aplicación interactiva localmente desde la terminal:
```bash
streamlit run app_streamlit.py
```
